In [6]:
import re
import uuid
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

class LegalMDChunker:
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 100):
        self.chunk_size = chunk_size
        self.sub_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\d+\. ", "\n[a-z]\) ", "\n\n", "\n", " "]
        )

    def process_file(self, file_path: str):
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        article_pattern = r'(?=\n\*\*Điều \d+\\?\.)'
        articles = re.split(article_pattern, content)

        final_chunks = []
        for art in articles:
            art = art.strip()
            if not art: continue

            lines = art.split('\n')
            article_title = lines[0].replace('**', '').replace('\\', '').strip()

            if len(art) <= self.chunk_size:
                final_chunks.append({
                    "content": art,
                    "metadata": {
                        "article_title": article_title,
                        "is_fragment": False,
                        "source": Path(file_path).name
                    }
                })
            else:
                sub_texts = self.sub_splitter.split_text(art)
                for i, sub in enumerate(sub_texts):
                    context_content = sub if i == 0 else f"{article_title} (Tiếp theo):\n{sub}"
                    final_chunks.append({
                        "content": context_content,
                        "metadata": {
                            "article_title": article_title,
                            "is_fragment": True,
                            "chunk_index": i,
                            "source": Path(file_path).name
                        }
                    })
        return final_chunks

chunker = LegalMDChunker(chunk_size=800)
chunks = chunker.process_file("data/Bo-luat-hinh-su.doc.md")

print(f"Đã xử lý xong. Tổng số chunks: {len(chunks)}")
print(f"Ví dụ nội dung chunk 1:\n{chunks[189]['content']}")

Đã xử lý xong. Tổng số chunks: 986
Ví dụ nội dung chunk 1:
**Điều 119\. Tội chống phá cơ sở giam giữ**

1\. Người nào nhằm chống chính quyền nhân dân mà phá cơ sở giam giữ, tổ chức trốn khỏi cơ sở giam giữ, đánh tháo người bị giam giữ, người bị áp giải hoặc trốn khỏi cơ sở giam giữ, thì bị phạt tù từ 10 năm đến 20 năm hoặc tù chung thân.

2\. Phạm tội trong trường hợp ít nghiêm trọng, thì bị phạt tù từ 03 năm đến 10 năm.

3\. Người chuẩn bị phạm tội này, thì bị phạt tù từ 01 năm đến 05 năm.


In [ ]:
import re
from pathlib import Path

def clean_and_convert_blhs(input_path: str, output_path: str):
    with open(input_path, "r", encoding="utf-8") as f:
        content = f.read()

    content = re.sub(r'\|.*\|', '', content)
    content = re.sub(r'[\-\:]{3,}', '', content)

    chapter_pattern = r'^.*(Phần|Chương|Mục)\s+[IVX\d]+.*$'
    content = re.sub(chapter_pattern, '', content, flags=re.MULTILINE | re.IGNORECASE)
    
    upper_title_pattern = r'^[A-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠƯ\s]+$'
    content = re.sub(upper_title_pattern, '', content, flags=re.MULTILINE)

    content = content.replace('**', '')
    content = content.replace('\\.', '.')
    content = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', content)

    content = re.sub(r'\n\s*\n\s*\n+', '\n\n', content)
    lines = [line.strip() for line in content.split('\n')]
    clean_content = '\n'.join([l for l in lines if l])

    final_text = re.sub(r'(Điều \d+)', r'\n\1', clean_content).strip()

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    print(f"Đã dọn dẹp xong! File lưu tại: {output_path}")

clean_and_convert_blhs("data/Bo-luat-hinh-su.doc.md", "blhs_final_clean.txt")

✅ Đã dọn dẹp xong! File lưu tại: blhs_final_clean.txt


In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), 'src'))

from chunking import ChunkingStrategyComparator

with open('blhs_final_clean.txt', 'r', encoding='utf-8') as f:
    text = f.read()

comparator = ChunkingStrategyComparator()
results = comparator.compare(text, chunk_size=500)

for name, data in results.items():
    print(f"Strategy: {name}")
    print(f"  - Chunk Count: {data['count']}")
    print(f"  - Avg Length: {data['avg_length']}")

Strategy: fixed_size
  - Chunk Count: 1147
  - Avg Length: 499.85
Strategy: by_sentences
  - Chunk Count: 1108
  - Avg Length: 495.39
Strategy: recursive
  - Chunk Count: 1538
  - Avg Length: 356.57
